### Cleaning Up the columns with no information or duplicated information

In [72]:
import numpy as np
import pandas as pd

df = pd.read_csv("./data/weather/65 Years of Weather Data Bangladesh (1948 - 2013).csv")

# Drop columns with no information (row index) or duplicated information
clean = df.drop(columns=["Unnamed: 0", "Period", "X_COR", "Y_COR"])

# Move the station_id column to the front for easier reading
cols = clean.columns.tolist()
cols.insert(0, cols.pop(cols.index('Station Number')))
clean = clean[cols]

print(clean.shape)
clean.head()

(21120, 14)


,Station Number,Station Names,YEAR,Month,Max Temp,Min Temp,Rainfall,Relative Humidity,Wind Speed,Cloud Coverage,Bright Sunshine,LATITUDE,LONGITUDE,ALT
0,41950,Barisal,1949,1,29.4,12.3,0.0,68.0,0.453704,0.6,7.831915,22.7,90.36,4
1,41950,Barisal,1950,1,30.0,14.1,0.0,77.0,0.453704,0.8,7.831915,22.7,90.36,4
2,41950,Barisal,1951,1,28.2,12.3,0.0,77.0,0.453704,0.6,7.831915,22.7,90.36,4
3,41950,Barisal,1952,1,26.6,12.3,2.0,77.0,0.453704,1.0,7.831915,22.7,90.36,4
4,41950,Barisal,1953,1,30.0,13.3,10.0,75.0,0.453704,1.6,7.831915,22.7,90.36,4


In [73]:
# Rename to snake_case: lowercase, no spaces, easier to type
clean = clean.rename(columns={
    "Station Number": "st_id",
    "Station Names": "station",
    "YEAR": "year",
    "Month": "month",
    "Max Temp": "max_temp",
    "Min Temp": "min_temp",
    "Rainfall": "rainfall",
    "Relative Humidity": "humidity",
    "Wind Speed": "wind_speed",
    "Cloud Coverage": "cloud_cover",
    "Bright Sunshine": "sunshine",
    "LATITUDE": "lati",
    "LONGITUDE": "longi",
    "ALT": "alti",
})
clean.head()

,st_id,station,year,month,max_temp,min_temp,rainfall,humidity,wind_speed,cloud_cover,sunshine,lati,longi,alti
0,41950,Barisal,1949,1,29.4,12.3,0.0,68.0,0.453704,0.6,7.831915,22.7,90.36,4
1,41950,Barisal,1950,1,30.0,14.1,0.0,77.0,0.453704,0.8,7.831915,22.7,90.36,4
2,41950,Barisal,1951,1,28.2,12.3,0.0,77.0,0.453704,0.6,7.831915,22.7,90.36,4
3,41950,Barisal,1952,1,26.6,12.3,2.0,77.0,0.453704,1.0,7.831915,22.7,90.36,4
4,41950,Barisal,1953,1,30.0,13.3,10.0,75.0,0.453704,1.6,7.831915,22.7,90.36,4


In [74]:
# Altitude 0 is a missing-value sentinel, not a real altitude
clean["alti"] = clean["alti"].replace(0, np.nan)
print(clean.alti.isna().sum(), "rows with missing altitude")
# Show 10 random rows with missing altitude
clean[clean.alti.isna()].sample(5).head(n=5)

624 rows with missing altitude


,st_id,station,year,month,max_temp,min_temp,rainfall,humidity,wind_speed,cloud_cover,sunshine,lati,longi,alti
3776,41926,Chuadanga,1989,9,34.700000,25.8,321.444444,86.888889,0.900000,5.092308,5.144444,23.65,88.816670,NaN
2768,41977,Chittagong (City-Ambagan),2009,6,34.833333,25.8,742.000000,85.666667,3.666667,5.500000,6.411131,22.35,91.816600,NaN
19397,41858,Syedpur,2008,1,27.400000,11.7,32.000000,81.000000,1.100000,1.800000,5.100000,25.75,88.916667,NaN
19545,41858,Syedpur,1995,8,34.500000,26.0,300.000000,84.000000,1.300000,5.700000,5.377778,25.75,88.916667,NaN
3667,41926,Chuadanga,2005,4,40.700000,24.1,22.000000,70.000000,1.000000,2.500000,9.100000,23.65,88.816670,NaN


In [76]:
# Chronological order: station, then year, then month
clean = clean.sort_values(["station", "year", "month"]).reset_index(drop=True)

print(clean.shape)
print(clean["alti"].isna().sum(), "rows with missing altitude")
clean.head()

(21120, 14)
624 rows with missing altitude


,st_id,station,year,month,max_temp,min_temp,rainfall,humidity,wind_speed,cloud_cover,sunshine,lati,longi,alti
0,41950,Barisal,1949,1,29.4,12.3,0.0,68.0,0.453704,0.6,7.831915,22.7,90.36,4.0
1,41950,Barisal,1949,2,33.9,15.2,9.0,63.0,0.659259,0.9,8.314894,22.7,90.36,4.0
2,41950,Barisal,1949,3,36.7,20.2,8.0,59.0,1.085185,1.5,8.131915,22.7,90.36,4.0
3,41950,Barisal,1949,4,33.9,23.9,140.0,71.0,1.772222,3.9,8.219149,22.7,90.36,4.0
4,41950,Barisal,1949,5,35.6,25.0,217.0,76.0,1.703704,4.1,7.046809,22.7,90.36,4.0


### Total Number of Stations with Complete Features (Column Values)

In [87]:
rows_per_station = clean["station"].value_counts()
# print(rows_per_station)

full_stations = rows_per_station[rows_per_station == 66*12].index.tolist()
# print(full_stations)
print("Number of stations with complete data:", len(full_stations))

clean["full_coverage"] = clean["station"].isin(full_stations)
print("Number of rows with full coverage:", clean["full_coverage"].sum())
clean.sample(5).head()

Number of stations with complete data: 10
Number of rows with full coverage: 7920


,st_id,station,year,month,max_temp,min_temp,rainfall,humidity,wind_speed,cloud_cover,sunshine,lati,longi,alti,full_coverage
6327,41863,Dinajpur,1959,4,37.200000,22.600000,133.0,67.0,1.155556,1.200000,7.424000,25.63,88.66,37.0,True
9020,41907,Ishurdi,1975,9,32.800000,25.600000,177.0,84.0,2.800000,6.400000,5.220690,24.12,89.04,14.0,False
394,41950,Barisal,1981,11,32.800000,17.600000,0.0,76.0,0.300000,1.200000,8.900000,22.70,90.36,4.0,False
9510,41936,Jessore,1950,7,34.300000,25.700000,281.0,88.0,2.955556,6.300000,3.972340,23.17,89.22,7.0,True
11562,41989,Kutubdia,1978,7,32.893103,25.489655,697.0,88.4,2.900000,6.434483,4.016667,21.83,91.84,6.0,False


### 

In [91]:
amb = clean[clean["station"] == "Chittagong (City-Ambagan)"]
weather = ["max_temp", "min_temp", "rainfall", "humidity",
           "wind_speed", "cloud_cover", "sunshine"]


for col in weather:
    y2009 = amb[amb["year"] == 2009].set_index("month")[col]
    later_mean = amb[amb["year"].isin([2011, 2012, 2013])].groupby("month")[col].mean()
    gap = (y2009 - later_mean).abs().max()
    print(f"{col:12s} max gap = {gap:.6f}")
    
amb.head()

max_temp     max gap = 0.000000
min_temp     max gap = 0.000000
rainfall     max gap = 0.000000
humidity     max gap = 0.000000
wind_speed   max gap = 0.000000
cloud_cover  max gap = 0.800000
sunshine     max gap = 0.000000


,st_id,station,year,month,max_temp,min_temp,rainfall,humidity,wind_speed,cloud_cover,sunshine,lati,longi,alti,full_coverage
2748,41977,Chittagong (City-Ambagan),2009,1,31.566667,13.700000,0.000000,70.333333,3.033333,0.1,6.411131,22.35,91.8166,NaN,False
2749,41977,Chittagong (City-Ambagan),2009,2,34.833333,16.700000,0.333333,62.333333,3.166667,0.7,6.411131,22.35,91.8166,NaN,False
2750,41977,Chittagong (City-Ambagan),2009,3,35.366667,21.233333,23.666667,72.333333,3.800000,2.5,6.411131,22.35,91.8166,NaN,False
2751,41977,Chittagong (City-Ambagan),2009,4,35.433333,23.900000,135.333333,77.666667,3.733333,3.2,6.411131,22.35,91.8166,NaN,False
2752,41977,Chittagong (City-Ambagan),2009,5,34.666667,24.933333,312.666667,80.666667,3.900000,4.3,6.411131,22.35,91.8166,NaN,False
